# Satellite-Derived Drought Indicators as Early Predictors of Food Insecurity Risk in Northern Kenya

Reproducible pipeline for the paper. Every number and figure in the manuscript is
produced by the cells below.

The pipeline has eight stages:

1. County boundaries (Humanitarian Data Exchange, Kenya COD-AB)
2. Rainfall (CHIRPS v2.0 monthly, 1981 to 2026)
3. Standardized Precipitation Index at 1, 3, and 6 months
4. Vegetation (MODIS MOD13Q1 and MYD13Q1) and the Kogan Vegetation Condition Index
5. Outcome labels (FEWS NET acute food insecurity classifications, 2011 to 2026)
6. Modelling panel at 1, 3, and 6 months of lead time
7. XGBoost training, walk-forward evaluation, and baselines
8. SHAP analysis and figures

Stages 1 to 5 write county-level CSV files into `data/processed/`, which are
committed to the repository. When those files are present the notebook reads
them and runs stages 6 to 8 from scratch in a few minutes. When they are absent
the fetch functions re-download from the public archives, which takes roughly an
hour and about three gigabytes of transfer.

Author: Craig Carlos Ouma, Independent Researcher.

## 0. Setup

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

import config
from config import PROCESSED, STUDY_COUNTIES, TRAIN_END_YEAR, TEST_START_YEAR

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

print("study counties:", ", ".join(STUDY_COUNTIES))
print("rainfall window:", config.CHIRPS_START_YEAR, "to", config.CHIRPS_END_YEAR)
print("vegetation window:", config.MODIS_START_YEAR, "to", config.MODIS_END_YEAR)
print("fixed split: train through", TRAIN_END_YEAR, "| test from", TEST_START_YEAR)

study counties: Turkana, Marsabit, Samburu, Baringo, Wajir
rainfall window: 1981 to 2026
vegetation window: 2001 to 2026
fixed split: train through 2020 | test from 2021


## 1. County boundaries

Kenya's 47 counties from the Humanitarian Data Exchange common operational
dataset. No account or token is required. Earth observation indicators are
extracted for all 47 so the classifier has a national panel to learn from; the
five arid and semi-arid study counties are analysed separately throughout.

In [2]:
from fetch_boundaries import load_counties, load_target_counties

counties = load_counties()
targets = load_target_counties()

print(f"{len(counties)} counties total, {len(targets)} study counties")
targets[["county", "pcode", "area_sqkm"]].assign(
    area_sqkm=lambda d: d["area_sqkm"].round(0)
)

47 counties total, 5 study counties


,county,pcode,area_sqkm
0,Turkana,KE023,70132.0
1,Marsabit,KE010,76028.0
2,Samburu,KE025,21016.0
3,Baringo,KE030,10889.0
4,Wajir,KE008,56669.0


In [3]:
print(f"study county area: {targets['area_sqkm'].sum():,.0f} square kilometres")

study county area: 234,733 square kilometres


## 2. Rainfall: CHIRPS v2.0

Monthly Africa rasters at 0.05 degrees, clipped to each county polygon and
reduced to a county mean. CHIRPS is in the public domain. Re-running the fetch
downloads roughly 550 rasters; the cached county series is read here.

In [4]:
from fetch_chirps import OUT_PATH as CHIRPS_PATH, build_table

if CHIRPS_PATH.exists():
    chirps = pd.read_csv(CHIRPS_PATH, parse_dates=["date"])
else:
    chirps = build_table(config.CHIRPS_START_YEAR, config.CHIRPS_END_YEAR)
    chirps.to_csv(CHIRPS_PATH, index=False)

print(f"{len(chirps):,} county-months, {chirps['county'].nunique()} counties, "
      f"{chirps['date'].min():%Y-%m} to {chirps['date'].max():%Y-%m}")

annual = (
    chirps[chirps["year"].between(1981, 2010)]
    .groupby(["county", "year"], as_index=False)["rainfall_mm"].sum()
    .groupby("county", as_index=False)["rainfall_mm"].mean()
)
annual[annual["county"].isin(STUDY_COUNTIES)].round(1)

25,756 county-months, 47 counties, 1981-01 to 2026-08


,county,rainfall_mm
0,Baringo,407.6
24,Marsabit,183.1
36,Samburu,233.4
42,Turkana,134.9
45,Wajir,220.9


## 3. Standardized Precipitation Index

Accumulated rainfall over 1, 3, and 6 months is fitted to a gamma distribution
per county and calendar month over the 1981 to 2010 World Meteorological
Organization normal period, then mapped through the inverse normal distribution.
Fixing the baseline to a period that ends before the modelling window keeps
test-period statistics out of the predictors.

In [5]:
from spi import OUT_PATH as SPI_PATH, build_spi_table

if SPI_PATH.exists():
    spi = pd.read_csv(SPI_PATH, parse_dates=["date"])
else:
    spi = build_spi_table()
    spi.to_csv(SPI_PATH, index=False)

print(f"{len(spi):,} county-months")
spi[["spi1", "spi3", "spi6"]].describe().round(3)

25,756 county-months


,spi1,spi3,spi6
count,25756.000,25662.000,25521.000
mean,0.096,0.163,0.226
std,1.097,1.125,1.157
min,-3.665,-4.051,-3.496
25%,-0.697,-0.631,-0.561
50%,0.005,0.095,0.140
75%,0.800,0.846,0.888
max,4.753,4.753,4.753


In [6]:
# The driest county-months on the six-month index, within the study counties.
study_spi = spi[spi["county"].isin(STUDY_COUNTIES)]
study_spi.nsmallest(8, "spi6")[["date", "county", "rainfall_mm", "spi3", "spi6"]].round(3)

/var/folders/hr/tjn1rbjx2f34m36bm34tlqjw0000gn/T/ipykernel_43208/3409527166.py:3: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  study_spi.nsmallest(8, "spi6")[["date", "county", "rainfall_mm", "spi3", "spi6"]].round(3)


,date,county,rainfall_mm,spi3,spi6
43,1984-08-01,Baringo,21.338,-1.890,-3.149
41,1984-06-01,Baringo,15.872,-2.842,-3.147
13386,2000-07-01,Marsabit,3.010,-1.366,-2.947
42,1984-07-01,Baringo,41.636,-2.629,-2.928
13385,2000-06-01,Marsabit,1.738,-2.689,-2.928
13387,2000-08-01,Marsabit,2.225,-0.964,-2.881
25024,2011-05-01,Wajir,7.757,-2.403,-2.750
44,1984-09-01,Baringo,20.786,-1.224,-2.704


## 4. Vegetation: MODIS NDVI and the Vegetation Condition Index

MOD13Q1 (Terra) and MYD13Q1 (Aqua), version 6.1, read as cloud-optimised
GeoTIFFs from the Microsoft Planetary Computer catalogue over the four MODIS
sinusoidal tiles covering Kenya. Composites are read at roughly 2 km rather than
the native 250 m; `fetch_ndvi.validate_decimation` checks that choice against a
full-resolution read.

VCI follows Kogan (1995), with the per-county, per-calendar-month minimum and
maximum taken over a 2001 to 2020 baseline that ends before the test window.

In [7]:
from fetch_ndvi import VCI_PATH, NDVI_PATH, add_vci, build_ndvi_table

if VCI_PATH.exists():
    vci = pd.read_csv(VCI_PATH, parse_dates=["date"])
else:
    monthly = build_ndvi_table(config.MODIS_START_YEAR, config.MODIS_END_YEAR)
    monthly.to_csv(NDVI_PATH, index=False)
    vci = add_vci(monthly)
    vci.to_csv(VCI_PATH, index=False)

print(f"{len(vci):,} county-months, {vci['date'].min():%Y-%m} to {vci['date'].max():%Y-%m}")
vci[vci["county"].isin(STUDY_COUNTIES)].groupby("county")[["ndvi", "vci"]].agg(
    ["mean", "min", "max"]
).round(2)

14,288 county-months, 2000-12 to 2026-08


ndvi                vci               
          mean   min   max   mean    min     max
county                                          
Baringo   0.50  0.29  0.68  50.65 -13.74  140.40
Marsabit  0.22  0.15  0.48  35.47 -10.76  188.41
Samburu   0.36  0.24  0.60  44.99  -9.41  138.67
Turkana   0.25  0.16  0.40  40.09   0.00  137.22
Wajir     0.28  0.19  0.60  39.49 -11.12  152.12

In [8]:
# Months absent from the MODIS record. These remove one FEWS NET reporting
# period (October 2025) from the modelling panel.
expected = pd.date_range("2001-01-01", vci["date"].max(), freq="MS")
present = set(vci[vci["county"] == "Turkana"]["date"])
missing = [f"{d:%Y-%m}" for d in expected if d not in present]
print("missing months:", missing)

missing months: ['2025-07', '2025-08', '2025-09', '2025-10', '2026-06']


## 5. Outcome: FEWS NET acute food insecurity classifications

These are the classifications FEWS NET publishes for Kenya on the Integrated
Food Security Phase Classification scale, obtained from the public FEWS NET Data
Warehouse endpoints that the Humanitarian Data Exchange lists. They are real
published classifications, not a proxy constructed from the predictors.

FEWS NET classifies sub-county units, so for each reporting date the dissolved
per-phase polygons are intersected with county boundaries in an equal-area
projection. A county's label is the phase covering the largest share of its
classified area.

In [9]:
from fetch_ipc import OUT_PATH as IPC_PATH, build_county_outcomes

if IPC_PATH.exists():
    ipc = pd.read_csv(IPC_PATH, parse_dates=["reporting_date"])
else:
    ipc = build_county_outcomes()
    ipc.to_csv(IPC_PATH, index=False)

print(f"{len(ipc):,} county-periods, {ipc['reporting_date'].nunique()} reporting dates, "
      f"{ipc['reporting_date'].min():%Y-%m} to {ipc['reporting_date'].max():%Y-%m}")
print(f"classified area share: mean {ipc['classified_share'].mean():.3f}, "
      f"minimum {ipc['classified_share'].min():.3f}")
ipc["ipc_phase_name"].value_counts().to_frame("county_periods")

2,350 county-periods, 50 reporting dates, 2011-01 to 2026-06
classified area share: mean 0.971, minimum 0.663


,county_periods
ipc_phase_name,
Minimal,1401
Stressed,749
Crisis,190
Emergency,10


In [10]:
study_ipc = ipc[ipc["county"].isin(STUDY_COUNTIES)]
national_crisis = (ipc["ipc_phase"] >= 3).mean()
study_crisis = (study_ipc["ipc_phase"] >= 3).mean()
print(f"Crisis or worse: {study_crisis:.1%} of study county-periods, "
      f"{national_crisis:.1%} nationally")

Crisis or worse: 32.8% of study county-periods, 8.5% nationally


## 6. Modelling panel

One row per county and reporting date. At a lead of L months the most recent
month a feature may use is L months before the month being classified, so the
panel poses a forecasting question rather than a nowcasting one. Phases 3, 4,
and 5 are collapsed into one Crisis-or-worse class.

In [11]:
from build_dataset import (
    CLASS_NAMES,
    FEATURES,
    FEATURES_NO_COUNTY,
    FEATURES_WITH_PERSISTENCE,
    OUT_TEMPLATE,
    build_panel,
)

panels = {}
for lead in (1, 3, 6):
    panel = build_panel(lead)
    panel.to_csv(PROCESSED / OUT_TEMPLATE.format(lead=lead), index=False)
    panels[lead] = panel
    counts = panel.groupby("split")["ipc_class"].value_counts().unstack(fill_value=0)
    print(f"lead {lead}m: {len(panel):,} county-periods")
    print(counts.rename(columns=CLASS_NAMES).to_string(), "\n")

print("features:", ", ".join(FEATURES))

lead 1m: 2,303 county-periods
ipc_class  Minimal  Stressed  Crisis or worse
split                                        
test           359       190              109
train         1018       539               88 



lead 3m: 2,256 county-periods
ipc_class  Minimal  Stressed  Crisis or worse
split                                        
test           335       179               97
train         1018       539               88 



lead 6m: 2,256 county-periods
ipc_class  Minimal  Stressed  Crisis or worse
split                                        
test           335       184               92
train         1018       539               88 

features: spi1_recent, spi3_recent, spi6_recent, spi3_prior, vci_recent, vci_prior1, vci_prior2, vci_trend, mean_annual_rain_mm, month, season, county


## 7. Model, baselines, and evaluation

XGBoost with class weights inversely proportional to class frequency. Two
evaluation schemes are run: a single fixed split (train through 2020, test from
2021) and expanding-window walk-forward validation, one fold per year from 2016,
which the paper treats as primary.

Three variants are fitted. The Earth-observation-only model is the one the paper
interprets. A second adds the previous published classification. A third drops
county identity, as a sensitivity check on how much of the signal is satellite
data rather than memorised vulnerability.

Two baselines are reported: the majority class, and persistence, meaning a rule
that repeats each county's previous classification.

In [12]:
from train_model import (
    MODEL_DIR,
    RESULTS_PATH,
    common_test_keys,
    run,
    walk_forward,
)

variants = [
    ("eo_only", FEATURES),
    ("eo_plus_persistence", FEATURES_WITH_PERSISTENCE),
    ("eo_no_county", FEATURES_NO_COUNTY),
]

shared_keys = common_test_keys([1, 3, 6])
results = {"common_test_n": len(shared_keys)}

for tag, features in variants:
    for lead in (1, 3, 6):
        results[f"{tag}_lead_{lead}"] = run(lead, features, tag, shared_keys)

print(f"fixed split, common test block of {len(shared_keys):,} county-periods\n")
rows = []
for tag, _ in variants:
    for lead in (1, 3, 6):
        m = results[f"{tag}_lead_{lead}"]["model"]
        rows.append({
            "variant": tag, "lead": lead,
            "accuracy": round(m["accuracy"], 3),
            "macro_f1": round(m["macro_f1"], 3),
            "auroc": round(m.get("auroc_ovr_macro", float("nan")), 3),
            "crisis_recall": round(m["per_class"]["Crisis or worse"]["recall"], 3),
        })
pd.DataFrame(rows)

/Users/craigouma/Documents/eo-drought-food-security-kenya/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/Users/craigouma/Documents/eo-drought-food-security-kenya/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/Users/craigouma/Documents/eo-drought-food-security-kenya/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/Users/craigouma/Documents/eo-drought-food-security-kenya/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/Users/craigouma/Documents/eo-drought-food-security-kenya/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


...

/Users/craigouma/Documents/eo-drought-food-security-kenya/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/Users/craigouma/Documents/eo-drought-food-security-kenya/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


fixed split, common test block of 564 county-periods



/Users/craigouma/Documents/eo-drought-food-security-kenya/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,variant,lead,accuracy,macro_f1,auroc,crisis_recall
0,eo_only,1,0.778,0.681,0.910,0.360
1,eo_only,3,0.812,0.726,0.916,0.393
2,eo_only,6,0.832,0.749,0.916,0.416
3,eo_plus_persistence,1,0.816,0.719,0.942,0.393
4,eo_plus_persistence,3,0.842,0.753,0.946,0.416
5,eo_plus_persistence,6,0.858,0.772,0.943,0.438
6,eo_no_county,1,0.755,0.653,0.882,0.360
7,eo_no_county,3,0.794,0.702,0.897,0.382
8,eo_no_county,6,0.819,0.747,0.898,0.472


In [13]:
for tag, features in variants:
    for lead in (1, 3, 6):
        pooled, summary = walk_forward(lead, features)
        results[f"walkforward_{tag}_lead_{lead}"] = summary
        if tag == "eo_only":
            pooled.to_csv(
                PROCESSED / f"walkforward_predictions_lead{lead}.csv", index=False
            )

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_PATH.write_text(json.dumps(results, indent=2))

walk = results["walkforward_eo_only_lead_1"]
print(f"walk-forward: {walk['n_folds']} folds, {walk['n_test']:,} pooled "
      f"out-of-sample county-periods, {walk['test_years'][0]} to {walk['test_years'][1]}")

walk-forward: 10 folds, 1,363 pooled out-of-sample county-periods, 2016 to 2026


In [14]:
# Headline comparison, pooled walk-forward predictions. This is Table 2 of the paper.
def summarise(key, label):
    block = results[key]
    model = block["model"]
    return {
        "model": label,
        "n": model["n"],
        "accuracy": round(model["accuracy"], 3),
        "macro_f1": round(model["macro_f1"], 3),
        "auroc": round(model.get("auroc_ovr_macro", float("nan")), 3),
        "crisis_recall": round(model["per_class"]["Crisis or worse"]["recall"], 3),
    }

base = results["walkforward_eo_only_lead_1"]
table = [
    {"model": "majority class", "n": base["majority_baseline"]["n"],
     "accuracy": round(base["majority_baseline"]["accuracy"], 3),
     "macro_f1": round(base["majority_baseline"]["macro_f1"], 3),
     "auroc": float("nan"), "crisis_recall": 0.0},
    {"model": "persistence", "n": base["persistence_baseline"]["n"],
     "accuracy": round(base["persistence_baseline"]["accuracy"], 3),
     "macro_f1": round(base["persistence_baseline"]["macro_f1"], 3),
     "auroc": float("nan"),
     "crisis_recall": round(
         base["persistence_baseline"]["per_class"]["Crisis or worse"]["recall"], 3)},
    summarise("walkforward_eo_only_lead_1", "satellite only, 1 month"),
    summarise("walkforward_eo_only_lead_3", "satellite only, 3 months"),
    summarise("walkforward_eo_only_lead_6", "satellite only, 6 months"),
    summarise("walkforward_eo_plus_persistence_lead_1", "satellite plus previous phase"),
    summarise("walkforward_eo_no_county_lead_1", "satellite without county identity"),
]
pd.DataFrame(table)

,model,n,accuracy,macro_f1,auroc,crisis_recall
0,majority class,1363,0.607,0.252,NaN,0.000
1,persistence,1363,0.850,0.769,NaN,0.609
2,"satellite only, 1 month",1363,0.810,0.707,0.913,0.397
3,"satellite only, 3 months",1316,0.809,0.717,0.911,0.424
4,"satellite only, 6 months",1316,0.809,0.716,0.912,0.463
5,satellite plus previous phase,1363,0.824,0.710,0.933,0.377
6,satellite without county identity,1363,0.775,0.664,0.896,0.358


### Where the classification changes

Persistence is correct on every stable period and wrong on every change, by
construction. Since a change is the event an early warning system exists to
anticipate, those periods are scored separately.

In [15]:
rows = []
for tag, _ in variants:
    for lead in (1, 3, 6):
        t = results[f"walkforward_{tag}_lead_{lead}"]["transitions"]
        rows.append({
            "variant": tag, "lead": lead,
            "changed": t["n_changed"],
            "model_exact_on_changed": round(t["model_exact_on_changed"], 3),
            "persistence_exact_on_changed": round(t["persistence_exact_on_changed"], 3),
            "deteriorations": t["n_worsened"],
            "direction_caught": round(t["model_direction_on_worsened"], 3),
            "ci": [round(x, 3) for x in t["model_direction_on_worsened_ci"]],
        })
pd.DataFrame(rows)

,variant,lead,changed,model_exact_on_changed,persistence_exact_on_changed,deteriorations,direction_caught,ci
0,eo_only,1,205,0.585,0.0,113,0.593,"[0.501, 0.679]"
1,eo_only,3,196,0.551,0.0,104,0.625,"[0.529, 0.712]"
2,eo_only,6,197,0.518,0.0,109,0.633,"[0.539, 0.718]"
3,eo_plus_persistence,1,205,0.434,0.0,113,0.416,"[0.329, 0.508]"
4,eo_plus_persistence,3,196,0.393,0.0,104,0.394,"[0.306, 0.49]"
5,eo_plus_persistence,6,197,0.350,0.0,109,0.358,"[0.274, 0.451]"
6,eo_no_county,1,205,0.561,0.0,113,0.540,"[0.448, 0.629]"
7,eo_no_county,3,196,0.622,0.0,104,0.625,"[0.529, 0.712]"
8,eo_no_county,6,197,0.513,0.0,109,0.615,"[0.521, 0.701]"


### The five study counties

In [16]:
study = results["walkforward_eo_only_lead_1"]["study_counties"]
model = study["model"]
transitions = study["transitions"]
print(f"pooled out-of-sample study county-periods: {study['n_test']}")
print(f"accuracy {model['accuracy']:.3f} against persistence "
      f"{study['persistence_baseline']['accuracy']:.3f}")
crisis = model["per_class"]["Crisis or worse"]
print(f"Crisis class: precision {crisis['precision']:.2f}, recall {crisis['recall']:.2f}, "
      f"support {int(crisis['support'])}")
print(f"on {transitions['n_changed']} changes: exact {transitions['model_exact_on_changed']:.3f}")
print(f"on {transitions['n_worsened']} deteriorations: direction caught "
      f"{transitions['model_direction_on_worsened']:.3f} "
      f"(95% interval {transitions['model_direction_on_worsened_ci'][0]:.2f} to "
      f"{transitions['model_direction_on_worsened_ci'][1]:.2f})")

pooled out-of-sample study county-periods: 145
accuracy 0.628 against persistence 0.669
Crisis class: precision 0.79, recall 0.46, support 65
on 48 changes: exact 0.583
on 27 deteriorations: direction caught 0.556 (95% interval 0.37 to 0.72)


### Crisis alert trade-off

In [17]:
curve = pd.DataFrame(results["walkforward_eo_only_lead_1"]["crisis_alert_curve"])
curve = curve[curve["threshold"].isin([0.10, 0.15, 0.20, 0.30, 0.40, 0.50])]
curve.assign(
    precision=lambda d: d["precision"].round(3),
    recall=lambda d: d["recall"].round(3),
).reset_index(drop=True)

,threshold,flagged,precision,recall
0,0.10,360,0.361,0.861
1,0.15,276,0.402,0.735
2,0.20,219,0.457,0.662
3,0.30,150,0.540,0.536
4,0.40,111,0.613,0.450
5,0.50,82,0.707,0.384


## 8. SHAP analysis

TreeExplainer on the Earth-observation-only model. Alongside global importance,
the mean SHAP contribution to the Crisis class is computed within bins of each
leading indicator, and the value at which it changes sign is recorded: the point
at which an indicator starts pushing the model toward a Crisis call rather than
away from it.

In [18]:
import shap_analysis

shap_summaries = {}
for lead in (1, 3, 6):
    shap_summaries[f"lead_{lead}"] = shap_analysis.run(lead)
shap_analysis.OUT_PATH.write_text(json.dumps(shap_summaries, indent=2))

importance = pd.DataFrame(shap_summaries["lead_1"]["global_importance"])
importance.round(3)

,feature,mean_abs_shap,mean_abs_shap_crisis
0,county,1.199,1.162
1,vci_prior2,0.183,0.344
2,spi6_recent,0.164,0.261
3,vci_recent,0.154,0.254
4,vci_prior1,0.178,0.244
5,mean_annual_rain_mm,0.300,0.166
6,season,0.077,0.141
7,spi1_recent,0.071,0.110
8,vci_trend,0.093,0.107
9,spi3_recent,0.063,0.099


In [19]:
thresholds = shap_summaries["lead_1"]["crisis_thresholds"]
for feature, result in thresholds.items():
    if result.get("threshold") is not None:
        print(f"{feature:<14} sign change at {result['threshold']:7.2f}  "
              f"({result['direction']})")

spi3_recent    sign change at    1.09  (above this value the indicator pushes toward Crisis)
vci_recent     sign change at   18.05  (below this value the indicator pushes toward Crisis)
vci_prior1     sign change at   18.19  (below this value the indicator pushes toward Crisis)
vci_prior2     sign change at   21.88  (below this value the indicator pushes toward Crisis)
spi6_recent    sign change at   -0.75  (below this value the indicator pushes toward Crisis)
vci_trend      sign change at   13.08  (above this value the indicator pushes toward Crisis)


## 9. Figures

All six figures in the paper, written to `paper/`.

In [20]:
import figures

figures.apply_style()
figures.county_map()
figures.rainfall_vci_timeseries()
figures.classification_performance()
figures.shap_bar()
figures.shap_beeswarm()
figures.shap_dependence()

for name in sorted(p.name for p in (ROOT / "paper").glob("fig_*.png")):
    print("wrote", name)

wrote fig_classification_performance.png
wrote fig_county_map.png
wrote fig_rainfall_vci_timeseries.png
wrote fig_shap_bar.png
wrote fig_shap_beeswarm.png
wrote fig_shap_dependence.png


## 10. Numbers quoted in the paper

A single cell reproducing the figures cited in the abstract and results, so any
discrepancy between the manuscript and the pipeline is visible.

In [21]:
walk = results["walkforward_eo_only_lead_1"]
model = walk["model"]
transitions = walk["transitions"]
persistence = walk["persistence_baseline"]
majority = walk["majority_baseline"]

print(f"pooled out-of-sample county-periods      {model['n']:,}")
print(f"satellite-only accuracy                  {model['accuracy']:.3f} "
      f"({model['accuracy_ci'][0]:.3f} to {model['accuracy_ci'][1]:.3f})")
print(f"satellite-only macro F1                  {model['macro_f1']:.3f}")
print(f"satellite-only AUROC                     {model['auroc_ovr_macro']:.3f}")
print(f"persistence accuracy                     {persistence['accuracy']:.3f}")
print(f"majority-class accuracy                  {majority['accuracy']:.3f}")
print(f"periods where the phase changed          {transitions['n_changed']}")
print(f"model exact on those                     {transitions['model_exact_on_changed']:.3f}")
print(f"deteriorations                           {transitions['n_worsened']}")
print(f"direction caught, 1 month lead           {transitions['model_direction_on_worsened']:.3f}")
print(f"direction caught, 6 month lead           "
      f"{results['walkforward_eo_only_lead_6']['transitions']['model_direction_on_worsened']:.3f}")
print(f"direction caught, plus previous phase    "
      f"{results['walkforward_eo_plus_persistence_lead_1']['transitions']['model_direction_on_worsened']:.3f}")
print(f"VCI sign change                          "
      f"{shap_summaries['lead_1']['crisis_thresholds']['vci_recent']['threshold']:.1f}")
print(f"SPI-6 sign change                        "
      f"{shap_summaries['lead_1']['crisis_thresholds']['spi6_recent']['threshold']:.2f}")

pooled out-of-sample county-periods      1,363
satellite-only accuracy                  0.810 (0.788 to 0.830)
satellite-only macro F1                  0.707
satellite-only AUROC                     0.913
persistence accuracy                     0.850
majority-class accuracy                  0.607
periods where the phase changed          205
model exact on those                     0.585
deteriorations                           113
direction caught, 1 month lead           0.593
direction caught, 6 month lead           0.633
direction caught, plus previous phase    0.416
VCI sign change                          18.0
SPI-6 sign change                        -0.75


## Reproducibility

Random seeds are fixed in `src/train_model.py`. Model results are written to
`model/model_results.json`, SHAP summaries to `model/shap_summary.json`, and
per-fold out-of-sample predictions to
`data/processed/walkforward_predictions_lead*.csv`. `DECISIONS.md` records the
substantive choices, including the four departures from the original analysis
plan and the checks run on each.